# Universal Animation Engine — L4 24 GB + Low-Disk Colab

Fresh-runtime animation notebook:

```text
Humanoid 3D
  ↓
Make-It-Animatable rig provider
  ↓
ARDY text motion
  ↓
ARDY Motion Adapter
  ↓
Canonical Motion
  ↓
Universal Retarget Core
  ↓
MIA/Mixamo Rig Adapter
  ↓
IK / Contact extension layer
  ↓
Strict deformation + motion validation
  ↓
FBX / GLB preview / Unreal ZIP
```

**Active V1:** ARDY Core + MIA + `mia_mixamo` preset. Auto-Rig-Pro is not used for motion transfer. ARDY contact channels are preserved, while the V1 IK/contact stage is still a no-edit extension point. TRELLIS is intentionally not installed here. Run top to bottom in a fresh **L4 24 GB** Colab runtime.


In [ ]:
# 1. Fresh runtime setup + GPU/disk + Git clone
import os, sys, json, time, pathlib, shutil, subprocess, collections

LOG_DIR = pathlib.Path('/content/engine_logs'); LOG_DIR.mkdir(parents=True, exist_ok=True)

def disk_status(label='disk', minimum_free_gib=None):
    total, used, free = shutil.disk_usage('/content'); GiB = 1024**3
    print(f'[DISK] {label}: used={used/GiB:.1f} GiB | free={free/GiB:.1f} GiB | total={total/GiB:.1f} GiB', flush=True)
    if minimum_free_gib is not None and free/GiB < minimum_free_gib:
        raise RuntimeError(f'Only {free/GiB:.1f} GiB free; need at least {minimum_free_gib} GiB.')
    return free/GiB

def run_live(cmd, *, cwd=None, env=None, label='process'):
    cmd=[str(x) for x in cmd]; safe=''.join(ch if ch.isalnum() or ch in '-_' else '_' for ch in label)[:60]
    log_path=LOG_DIR/f"{time.strftime('%Y%m%d_%H%M%S')}_{safe}.log"
    print('\n'+'='*78); print('[RUN]',label); print('[CMD]',' '.join(cmd)); print('[LOG]',log_path); print('='*78, flush=True)
    started=time.time(); tail=collections.deque(maxlen=120)
    with log_path.open('w',encoding='utf-8',errors='replace') as log:
        proc=subprocess.Popen(cmd,cwd=str(cwd) if cwd else None,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout:
            print(line,end='',flush=True); log.write(line); log.flush(); tail.append(line.rstrip())
        rc=proc.wait()
    elapsed=time.time()-started
    if rc!=0:
        print('\n'+'!'*78); print(f'[FAILED] {label} | exit={rc} | elapsed={elapsed/60:.1f} min'); print('[FULL LOG]',log_path); print('[LAST LOG LINES]')
        for line in tail: print(line)
        print('!'*78); raise RuntimeError(f'{label} failed with exit code {rc}. See {log_path}.')
    print(f'\n[DONE] {label} | elapsed={elapsed/60:.1f} min', flush=True); return log_path

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('No NVIDIA GPU. Choose Runtime → Change runtime type → L4, then reconnect.')
smi=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader,nounits'],text=True,capture_output=True,check=True).stdout.strip().splitlines()
gpu_name,memory_mib=[x.strip() for x in smi[0].rsplit(',',1)]; memory_mib=int(memory_mib)
print(f'[GPU] {gpu_name} | VRAM={memory_mib/1024:.1f} GiB')
if memory_mib < 22000: raise RuntimeError('Use >=22 GiB VRAM. L4 24 GB is recommended; T4 16 GB is too tight for default ARDY Llama/LLM2Vec.')
disk_status('fresh runtime', minimum_free_gib=65)

REPO=pathlib.Path('/content/My-works')
if REPO.exists(): shutil.rmtree(REPO)
run_live(['git','clone','--progress','--depth','1','https://github.com/Logan17de/My-works.git',str(REPO)],label='Clone My-works')
ENGINE_ROOT=REPO/'ai-3d-animation-engines'; TOOLS_ANIM=ENGINE_ROOT/'animation-engine'
GIT_HEAD=subprocess.run(['git','-C',str(REPO),'rev-parse','--short','HEAD'],text=True,capture_output=True,check=True).stdout.strip()
print('[GIT] HEAD:',GIT_HEAD); print('[ENGINE]',TOOLS_ANIM)


## 2. Optional Google Drive build cache
Caches pinned source snapshots, compiled/native wheels, and small downloads. Large ARDY/Llama/MIA runtime weights remain local to this Colab session.


In [ ]:
USE_DRIVE_BUILD_CACHE = True #@param {type:"boolean"}
DRIVE_CACHE_ROOT = '/content/drive/MyDrive/AI3D_Engine_Cache' #@param {type:"string"}
if USE_DRIVE_BUILD_CACHE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    cache_root=pathlib.Path(DRIVE_CACHE_ROOT)
    for name in ('sources','wheels','downloads'): (cache_root/name).mkdir(parents=True,exist_ok=True)
    os.environ['ENGINE_CACHE_ROOT']=str(cache_root); print('[CACHE] Enabled:',cache_root)
else:
    os.environ.pop('ENGINE_CACHE_ROOT',None); print('[CACHE] Disabled')


## 3. Hugging Face authentication + access precheck
The token must have access to **`jasongzy/Mixamo`** and **`meta-llama/Meta-Llama-3-8B-Instruct`**. Save a READ token in Colab Secrets as `HF_TOKEN`, or enter it when prompted.


In [ ]:
import getpass
from google.colab import userdata
subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir','huggingface_hub','hf_xet'],check=True)
from huggingface_hub import HfApi, hf_hub_download
HF_TOKEN=None
try: HF_TOKEN=userdata.get('HF_TOKEN')
except Exception: pass
if not HF_TOKEN: HF_TOKEN=getpass.getpass('Hugging Face READ token (hidden): ').strip()
if not HF_TOKEN: raise RuntimeError('HF_TOKEN is required.')
os.environ['HF_TOKEN']=HF_TOKEN; os.environ['HF_HOME']='/content/huggingface'; os.environ['HF_XET_HIGH_PERFORMANCE']='1'
api=HfApi(token=HF_TOKEN); who=api.whoami(token=HF_TOKEN); print('[HF] Logged in as:',who.get('name') or who.get('fullname') or 'authenticated user')
print('[HF] Checking jasongzy/Mixamo access...',flush=True)
mixamo_files=api.list_repo_files('jasongzy/Mixamo',repo_type='dataset',token=HF_TOKEN)
bone=next((f for f in mixamo_files if pathlib.PurePosixPath(f).name.startswith('bones') and f.endswith('.fbx')),None)
if not bone: raise RuntimeError('Mixamo access succeeded but no bones*.fbx template was found.')
hf_hub_download('jasongzy/Mixamo',filename=bone,repo_type='dataset',token=HF_TOKEN,local_dir='/tmp/hf_access_check_mixamo')
print('[HF] ✅ Mixamo access OK')
print('[HF] Checking Meta-Llama-3-8B-Instruct access...',flush=True)
hf_hub_download('meta-llama/Meta-Llama-3-8B-Instruct',filename='config.json',token=HF_TOKEN,local_dir='/tmp/hf_access_check_llama')
print('[HF] ✅ Meta Llama access OK')
shutil.rmtree('/tmp/hf_access_check_mixamo',ignore_errors=True); shutil.rmtree('/tmp/hf_access_check_llama',ignore_errors=True)


## 4. Install Animation Engine — low-disk mode
Installs ARDY + Make-It-Animatable + Blender integration only. TRELLIS stays out of this runtime.


In [ ]:
installer=TOOLS_ANIM/'install_animation_low_disk.sh'
run_live(['bash','-n',str(installer)],label='Animation installer syntax check')
install_env=os.environ.copy(); install_env['HF_TOKEN']=HF_TOKEN; install_env['HF_HOME']='/content/huggingface'; install_env['HF_XET_HIGH_PERFORMANCE']='1'
run_live(['bash',str(installer)],env=install_env,label='Animation Engine low-disk installation')
disk_status('after installation cleanup',minimum_free_gib=32)


## 5. Universal Motion Engine preflight
Checks the Git code before loading expensive models. Active V1 is **ARDY → Canonical Motion → Universal Retarget Core → `mia_mixamo` adapter → validator**. Rigify/UE5/MetaHuman are extension points, not active presets yet.


In [ ]:
subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
GIT_HEAD=subprocess.run(['git','-C',str(REPO),'rev-parse','--short','HEAD'],text=True,capture_output=True,check=True).stdout.strip(); print('[GIT] Running HEAD:',GIT_HEAD)
required=[TOOLS_ANIM/'run_universal_animation_pipeline.py',TOOLS_ANIM/'build_canonical_motion.py',TOOLS_ANIM/'run_universal_retarget.py',TOOLS_ANIM/'validate_animation_contract.py',TOOLS_ANIM/'universal_motion/model.py',TOOLS_ANIM/'universal_motion/retarget.py',TOOLS_ANIM/'universal_motion/ik.py',TOOLS_ANIM/'universal_motion/sources/ardy.py',TOOLS_ANIM/'universal_motion/rigs/base.py',TOOLS_ANIM/'universal_motion/rigs/mia.py',TOOLS_ANIM/'universal_motion/rigs/registry.py',TOOLS_ANIM/'universal_motion/rig_presets/mia_mixamo.json']
for p in required:
    if not p.is_file(): raise FileNotFoundError(f'Universal engine file missing: {p}')
py_files=[TOOLS_ANIM/'run_universal_animation_pipeline.py',TOOLS_ANIM/'build_canonical_motion.py',TOOLS_ANIM/'run_universal_retarget.py',*sorted((TOOLS_ANIM/'universal_motion').rglob('*.py'))]
for p in py_files: subprocess.run([sys.executable,'-m','py_compile',str(p)],check=True)
preset=json.loads((TOOLS_ANIM/'universal_motion/rig_presets/mia_mixamo.json').read_text(encoding='utf-8'))
if preset.get('status')!='active': raise RuntimeError('mia_mixamo preset is not active.')
if 'hips' not in preset.get('semantic_bones',{}) or 'head' not in preset.get('semantic_bones',{}): raise RuntimeError('mia_mixamo preset is incomplete.')
print(f"[ENGINE] ✅ Syntax OK | preset={preset['id']} | provider={preset['provider']} | semantic bones={len(preset['semantic_bones'])}")
print('MotionSource : ARDY Core'); print('Canonical    : semantic humanoid motion'); print('TargetRig    : mia_mixamo'); print('Retarget     : universal rest-space core'); print('ARP transfer : disabled'); print('IK/contact   : separate V1 extension layer (no edits yet)')


## 6. Upload humanoid + choose motion
Upload exactly one `.glb`, `.fbx`, `.obj`, or polygon `.ply`. For TRELLIS output, use the scaled character asset from the 3D notebook.


In [ ]:
from google.colab import files
uploaded=files.upload()
if len(uploaded)!=1: raise ValueError('Upload exactly one humanoid GLB/FBX/OBJ/PLY.')
TARGET_CHARACTER=f"/content/{next(iter(uploaded))}"
if pathlib.Path(TARGET_CHARACTER).suffix.lower() not in {'.glb','.fbx','.obj','.ply'}: raise ValueError('Unsupported character format.')
PROMPT = 'A person walks forward, stops, and waves with the right hand.' #@param {type:"string"}
DURATION_SECONDS = 6.0 #@param {type:"number"}
SEED = 0 #@param {type:"integer"}
TARGET_ALREADY_RIGGED = False #@param {type:"boolean"}
MIA_NO_FINGERS = True #@param {type:"boolean"}
MOTION_SOURCE = 'ardy' #@param ['ardy']
RIG_PRESET = 'mia_mixamo' #@param ['mia_mixamo']
OUTPUT_ANIM_DIR='/content/animation_outputs'
print('Character      :',TARGET_CHARACTER); print('Prompt         :',PROMPT); print('Duration       :',DURATION_SECONDS,'sec'); print('Seed           :',SEED); print('Already rigged :',TARGET_ALREADY_RIGGED); print('No fingers     :',MIA_NO_FINGERS); print('Motion source  :',MOTION_SOURCE); print('Rig preset     :',RIG_PRESET)
disk_status('before ARDY/MIA runtime downloads',minimum_free_gib=28)


## 7. Run complete Universal Animation Engine
Runs ARDY generation, canonical conversion, MIA rigging, universal retargeting, the separate contact/IK extension stage, strict skinned-deformation/motion validation, and Unreal packaging. The ZIP is produced only after the strict contract passes.


In [ ]:
subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
cmd=['python',str(TOOLS_ANIM/'run_universal_animation_pipeline.py'),'--character',TARGET_CHARACTER,'--prompt',PROMPT,'--duration',str(DURATION_SECONDS),'--seed',str(SEED),'--output-dir',OUTPUT_ANIM_DIR,'--motion-source',MOTION_SOURCE,'--rig-preset',RIG_PRESET]
if TARGET_ALREADY_RIGGED: cmd.append('--already-rigged')
if MIA_NO_FINGERS: cmd.append('--no-fingers')
runtime_env=os.environ.copy(); runtime_env['HF_TOKEN']=HF_TOKEN; runtime_env['HF_HOME']='/content/huggingface'; runtime_env['HF_XET_HIGH_PERFORMANCE']='1'; runtime_env['TEXT_ENCODER_MODE']='local'; runtime_env['MPLBACKEND']='Agg'
run_live(cmd,env=runtime_env,label='Universal Animation Engine')
disk_status('after Universal Animation Engine')
RAW_MOTION=f'{OUTPUT_ANIM_DIR}/motion.npz'; MOTION_BRIDGE=f'{OUTPUT_ANIM_DIR}/motion_bridge.npz'; CANONICAL_MOTION=f'{OUTPUT_ANIM_DIR}/canonical_motion.npz'; MOTION_PREVIEW=f'{OUTPUT_ANIM_DIR}/motion_preview.mp4'; RIGGED_FBX=f'{OUTPUT_ANIM_DIR}/character_rigged.fbx'; FINAL_FBX=f'{OUTPUT_ANIM_DIR}/character_animated.fbx'; ANIMATED_PREVIEW=f'{OUTPUT_ANIM_DIR}/character_animated_preview.glb'; CONTRACT_REPORT=f'{OUTPUT_ANIM_DIR}/animation_contract_report.json'; PACKAGE_DIR=f'{OUTPUT_ANIM_DIR}/unreal_package'; ENGINE_MANIFEST=f'{PACKAGE_DIR}/universal_engine_manifest.json'; PACKAGE_ZIP=f'{OUTPUT_ANIM_DIR}/unreal_character_package.zip'
required_outputs=[RAW_MOTION,MOTION_BRIDGE,CANONICAL_MOTION,MOTION_PREVIEW,RIGGED_FBX,FINAL_FBX,CONTRACT_REPORT,ENGINE_MANIFEST,PACKAGE_ZIP]
for p in required_outputs:
    if not pathlib.Path(p).is_file(): raise RuntimeError(f'Missing Universal Animation output: {p}')
print('\n'+'='*78); print('✅ UNIVERSAL ANIMATION ENGINE COMPLETE'); print('='*78); print('Raw ARDY motion :',RAW_MOTION); print('Motion bridge   :',MOTION_BRIDGE); print('Canonical motion:',CANONICAL_MOTION); print('Motion preview  :',MOTION_PREVIEW); print('Rigged FBX      :',RIGGED_FBX); print('Animated FBX    :',FINAL_FBX)
if pathlib.Path(ANIMATED_PREVIEW).is_file(): print('Animated GLB    :',ANIMATED_PREVIEW)
print('Contract report :',CONTRACT_REPORT); print('Engine manifest :',ENGINE_MANIFEST); print('Unreal package  :',PACKAGE_ZIP); print('='*78)


## 8. Inspect validation + engine manifest
Shows exactly what the strict validator accepted and which universal-engine plugins produced the output. It does not weaken any gate.


In [ ]:
report_data=json.loads(pathlib.Path(CONTRACT_REPORT).read_text(encoding='utf-8')); manifest_data=json.loads(pathlib.Path(ENGINE_MANIFEST).read_text(encoding='utf-8'))
print('='*78); print('VALIDATION REPORT'); print('='*78); print(json.dumps(report_data,indent=2)[:20000]); print('\n'+'='*78); print('UNIVERSAL ENGINE MANIFEST'); print('='*78); print(json.dumps(manifest_data,indent=2))
if not report_data.get('passed'): raise RuntimeError('Contract report exists but passed=false.')


## 9. Preview generated motion
The MP4 is the source ARDY skeleton preview. `character_animated_preview.glb` is optional; the FBX is authoritative for skeletal animation.


In [ ]:
from IPython.display import Video, display
if pathlib.Path(MOTION_PREVIEW).is_file(): display(Video(MOTION_PREVIEW,embed=True,width=720))
else: print('Motion preview not found:',MOTION_PREVIEW)
print('Animated GLB preview:',ANIMATED_PREVIEW if pathlib.Path(ANIMATED_PREVIEW).is_file() else 'not produced (optional)')


## 10. Download outputs
Usually download the Unreal ZIP. Optional switches expose the final FBX, validation report, canonical motion, and source motion preview separately.


In [ ]:
from google.colab import files
DOWNLOAD_PACKAGE = True #@param {type:"boolean"}
DOWNLOAD_FINAL_FBX = False #@param {type:"boolean"}
DOWNLOAD_REPORT = False #@param {type:"boolean"}
DOWNLOAD_CANONICAL_MOTION = False #@param {type:"boolean"}
DOWNLOAD_MOTION_PREVIEW = False #@param {type:"boolean"}
if DOWNLOAD_PACKAGE: files.download(PACKAGE_ZIP)
if DOWNLOAD_FINAL_FBX: files.download(FINAL_FBX)
if DOWNLOAD_REPORT: files.download(CONTRACT_REPORT)
if DOWNLOAD_CANONICAL_MOTION: files.download(CANONICAL_MOTION)
if DOWNLOAD_MOTION_PREVIEW: files.download(MOTION_PREVIEW)


## 11. Troubleshooting — latest log
Run this after a failure. Each `run_live` failure also prints the exact full log path.


In [ ]:
def show_latest_log(lines=160):
    logs=sorted(LOG_DIR.glob('*.log'),key=lambda p:p.stat().st_mtime)
    if not logs: print('No engine logs found.'); return None
    latest=logs[-1]; print('[LATEST LOG]',latest); print('='*78)
    for line in latest.read_text(encoding='utf-8',errors='replace').splitlines()[-lines:]: print(line)
    return latest
LATEST_LOG=show_latest_log()


## 12. Optional final disk inspection
Use this after downloading your outputs, then disconnect the paid GPU runtime.


In [ ]:
disk_status('final runtime status'); print('\nLargest /content directories:'); subprocess.run(['bash','-lc','du -x -h -d 1 /content 2>/dev/null | sort -h | tail -20'],check=False)
